# Reddit API Data - 2026 Refresh
This notebook now uses public Reddit JSON feeds instead of Pushshift and private Reddit credentials.
It refreshes the latest `r/wallstreetbets` posts, extracts ticker mentions, and saves fresh CSV files for the rest of the project.

## Step 1 - Setup

In [1]:
from pathlib import Path
import json
import sys
from typing import cast
import pandas as pd
import seaborn as sns
try:
    ROOT = Path(__file__).resolve().parent
except NameError:
    ROOT = Path.cwd()
if not (ROOT / "src").exists():
    ROOT = Path.cwd()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from src.wsb_latest.pipeline import (
    annotate_posts_with_tickers,
    build_mentions_table,
    ensure_nltk_resources,
    load_recent_posts,
)
sns.set_theme(style="whitegrid")
pd.set_option("display.max_colwidth", 120)
ensure_nltk_resources()
print(f"Project root: {ROOT}")

Project root: C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets


## Step 2 - Notebook configuration

In [2]:
PER_FEED = 100
TOP_N = 10
OUTPUT_DIR = ROOT / "outputs" / "latest_wsb_analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Step 3 - Fetch the latest public WallStreetBets posts

In [3]:
posts_df, source_name = load_recent_posts(PER_FEED)
posts_df = cast(pd.DataFrame, posts_df)
print(json.dumps({"data_source": source_name, "posts": int(len(posts_df))}, indent=2))
posts_df.head(10)

{
  "data_source": "reddit_public_json",
  "posts": 166
}


,id,feed,title,body,score,num_comments,created_utc,permalink,url,text
12,1sqclc7,hot,Blue Orgin “accidentally” deploys their competitors satellites into the wrong orbit,"You can’t make this shit up, out of the hundreds of payloads in this rideshare rocket, they manage to fuck up $ASTS ...",56,14,2026-04-20 02:30:01+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sqclc7/blue_orgin_accidentally_deploys_their_competitors/,https://www.reddit.com/r/wallstreetbets/comments/1sqclc7/blue_orgin_accidentally_deploys_their_competitors/,"Blue Orgin “accidentally” deploys their competitors satellites into the wrong orbit You can’t make this shit up, out..."
8,1sqc8vd,hot,Jensen’s Leather Jacket is the only thing keeping the US Economy from a 1929 Speedrun. Change my mind.,"# How I Learned to Stop Worrying and Love the Bubble\n\nWe’ve officially reached the ""God Protocol"" phase of the mar...",108,44,2026-04-20 02:14:04+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sqc8vd/jensens_leather_jacket_is_the_only_thing_keeping/,https://www.reddit.com/r/wallstreetbets/comments/1sqc8vd/jensens_leather_jacket_is_the_only_thing_keeping/,Jensen’s Leather Jacket is the only thing keeping the US Economy from a 1929 Speedrun. Change my mind. # How I Learn...
5,1sqb7gi,hot,Straight of Hormuz smp prediction,,215,39,2026-04-20 01:26:43+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sqb7gi/straight_of_hormuz_smp_prediction/,https://i.redd.it/ugsgrcnoz8wg1.jpeg,Straight of Hormuz smp prediction
10,1sqayri,hot,[$MSTR] I have deemed it necessary to take out a personal loan to buy more.,,93,94,2026-04-20 01:15:48+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sqayri/mstr_i_have_deemed_it_necessary_to_take_out_a/,https://i.redd.it/yea5oihtx8wg1.png,[$MSTR] I have deemed it necessary to take out a personal loan to buy more.
4,1sqa6vz,hot,Market hasn’t even opened yet,Rape meeeeeee!!! Rape meeeeeee!!,436,158,2026-04-20 00:41:23+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sqa6vz/market_hasnt_even_opened_yet/,https://i.redd.it/208s5vrrr8wg1.jpeg,Market hasn’t even opened yet Rape meeeeeee!!! Rape meeeeeee!!
2,1sq9j1l,hot,Upcoming Stock Market Drop Will Be Epic Fury,**Fundamental Analysis:**\n\nEconomic backdrop - 🥭 does not care about high oil prices or the midterms. He wants hig...,1011,462,2026-04-20 00:11:27+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sq9j1l/upcoming_stock_market_drop_will_be_epic_fury/,https://www.reddit.com/r/wallstreetbets/comments/1sq9j1l/upcoming_stock_market_drop_will_be_epic_fury/,Upcoming Stock Market Drop Will Be Epic Fury **Fundamental Analysis:**\n\nEconomic backdrop - 🥭 does not care about ...
1,1sq3bkt,hot,"What Are Your Moves Tomorrow, April 20, 2026",This post contains content not supported on old Reddit. [Click here to view the full post](https://sh.reddit.com/r/w...,236,5818,2026-04-19 19:57:25+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sq3bkt/what_are_your_moves_tomorrow_april_20_2026/,https://www.reddit.com/r/wallstreetbets/comments/1sq3bkt/what_are_your_moves_tomorrow_april_20_2026/,"What Are Your Moves Tomorrow, April 20, 2026 This post contains content not supported on old Reddit. [Click here to ..."
6,1sq37v7,hot,I made my first M on friday.,,502,90,2026-04-19 19:53:29+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sq37v7/i_made_my_first_m_on_friday/,https://www.reddit.com/gallery/1sq37v7,I made my first M on friday.
14,1sq19dl,hot,I’d like to but a large Wendys combo on Monday,,88,67,2026-04-19 18:40:04+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sq19dl/id_like_to_but_a_large_wendys_combo_on_monday/,https://i.redd.it/n50nqc6bz6wg1.jpeg,I’d like to but a large Wendys combo on Monday
3,1sq14dm,hot,CNBC: Trump says talks between U.S. and Iran to resume in Pakistan on Monday,Just in time before the futures open. -2% day cancelled for Monday?,1612,326,2026-04-19 18:35:00+00:00,https://www.reddit.com/r/wallstreetbets/comments/1sq14dm/cnb

## Step 4 - Add sentiment and ticker extraction

In [4]:
annotated_posts_df = annotate_posts_with_tickers(posts_df)
annotated_posts_df = cast(pd.DataFrame, annotated_posts_df)
annotated_posts_df[["created_utc", "feed", "title", "score", "num_comments", "sentiment", "candidate_mentions"]].head(10)

,created_utc,feed,title,score,num_comments,sentiment,candidate_mentions
12,2026-04-20 02:30:01+00:00,hot,Blue Orgin “accidentally” deploys their competitors satellites into the wrong orbit,56,14,-0.3150,12
8,2026-04-20 02:14:04+00:00,hot,Jensen’s Leather Jacket is the only thing keeping the US Economy from a 1929 Speedrun. Change my mind.,108,44,-0.4754,2
5,2026-04-20 01:26:43+00:00,hot,Straight of Hormuz smp prediction,215,39,0.2263,0
10,2026-04-20 01:15:48+00:00,hot,[$MSTR] I have deemed it necessary to take out a personal loan to buy more.,93,94,0.0000,2
4,2026-04-20 00:41:23+00:00,hot,Market hasn’t even opened yet,436,158,-0.9112,0
2,2026-04-20 00:11:27+00:00,hot,Upcoming Stock Market Drop Will Be Epic Fury,1011,462,-0.9840,8
1,2026-04-19 19:57:25+00:00,hot,"What Are Your Moves Tomorrow, April 20, 2026",236,5818,-0.2411,0
6,2026-04-19 19:53:29+00:00,hot,I made my first M on friday.,502,90,0.0000,0
14,2026-04-19 18:40:04+00:00,hot,I’d like to but a large Wendys combo on Monday,88,67,0.1901,0
3,2026-04-19 18:35:00+00:00,hot,CNBC: Trump says talks between U.S. and Iran to resume in Pakistan on Monday,1612,326,-0.2500,1


## Step 5 - Rank the current top WallStreetBets tickers

In [5]:
summary_df, mentions_df = build_mentions_table(annotated_posts_df, TOP_N)
summary_df = cast(pd.DataFrame, summary_df)
mentions_df = cast(pd.DataFrame, mentions_df)
summary_df

,ticker,mention_count,post_count,avg_sentiment,total_score,total_comments,engagement,latest_mention,rank_score
48,SPY,24,13,-0.155254,12407,2443,133.623249,2026-04-20 00:11:27+00:00,76
31,MSTR,7,4,0.347100,562,193,31.395614,2026-04-20 01:15:48+00:00,23
43,RKLB,10,2,0.066450,1704,227,19.525002,2026-04-20 02:30:01+00:00,18
42,RACE,6,3,0.550033,2028,617,33.992138,2026-04-18 03:58:35+00:00,18
2,AMD,6,3,0.147533,8877,596,32.866942,2026-04-17 23:19:21+00:00,18
6,ASTS,6,3,-0.230967,511,285,26.778543,2026-04-20 02:30:01+00:00,18
50,TSLA,4,3,0.468600,998,518,26.855628,2026-04-18 10:58:22+00:00,16
41,QQQ,3,3,0.416500,3120,583,31.854319,2026-04-19 14:21:18+00:00,15
37,NVDA,3,3,-0.361900,993,250,28.905978,2026-04-20 02:14:04+00:00,15
35,NFLX,5,2,0.129350,44,57,12.336443,2026-04-17 21:02:22+00:00,13


## Step 6 - Save refreshed project CSV outputs
The notebook saves both the modern outputs under `outputs/latest_wsb_analysis/` and compatibility CSVs in the project root.

In [6]:
posts_export_df = annotated_posts_df.copy()
posts_export_df["timestamp"] = posts_export_df["created_utc"].dt.tz_convert(None)
posts_export_df["author"] = posts_export_df.get("author", "")
legacy_posts_df = posts_export_df[["title", "score", "body", "timestamp", "author"]].copy()
annotated_posts_df.to_csv(OUTPUT_DIR / "latest_wsb_posts.csv", index=False)
mentions_df.to_csv(OUTPUT_DIR / "latest_wsb_mentions.csv", index=False)
summary_df.to_csv(OUTPUT_DIR / "top10_wsb_stocks.csv", index=False)
legacy_posts_df.to_csv(ROOT / "wsb_reddit_api_data.csv", index=False)
legacy_posts_df.to_csv(ROOT / "wsb_pushshift_data.csv", index=False)
print("Saved files:")
for path in [
    OUTPUT_DIR / "latest_wsb_posts.csv",
    OUTPUT_DIR / "latest_wsb_mentions.csv",
    OUTPUT_DIR / "top10_wsb_stocks.csv",
    ROOT / "wsb_reddit_api_data.csv",
    ROOT / "wsb_pushshift_data.csv",
]:
    print(f"- {path}")

Saved files:
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\latest_wsb_posts.csv
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\latest_wsb_mentions.csv
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\outputs\latest_wsb_analysis\top10_wsb_stocks.csv
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\wsb_reddit_api_data.csv
- C:\Users\big_j\PycharmProjects\Project2-Sentiment-analysis-WSbets\wsb_pushshift_data.csv


## Step 7 - Preview the refreshed legacy-compatible CSV format

In [7]:
legacy_posts_df.head(10)

,title,score,body,timestamp,author
12,Blue Orgin “accidentally” deploys their competitors satellites into the wrong orbit,56,"You can’t make this shit up, out of the hundreds of payloads in this rideshare rocket, they manage to fuck up $ASTS ...",2026-04-20 02:30:01,
8,Jensen’s Leather Jacket is the only thing keeping the US Economy from a 1929 Speedrun. Change my mind.,108,"# How I Learned to Stop Worrying and Love the Bubble\n\nWe’ve officially reached the ""God Protocol"" phase of the mar...",2026-04-20 02:14:04,
5,Straight of Hormuz smp prediction,215,,2026-04-20 01:26:43,
10,[$MSTR] I have deemed it necessary to take out a personal loan to buy more.,93,,2026-04-20 01:15:48,
4,Market hasn’t even opened yet,436,Rape meeeeeee!!! Rape meeeeeee!!,2026-04-20 00:41:23,
2,Upcoming Stock Market Drop Will Be Epic Fury,1011,**Fundamental Analysis:**\n\nEconomic backdrop - 🥭 does not care about high oil prices or the midterms. He wants hig...,2026-04-20 00:11:27,
1,"What Are Your Moves Tomorrow, April 20, 2026",236,This post contains content not supported on old Reddit. [Click here to view the full post](https://sh.reddit.com/r/w...,2026-04-19 19:57:25,
6,I made my first M on friday.,502,,2026-04-19 19:53:29,
14,I’d like to but a large Wendys combo on Monday,88,,2026-04-19 18:40:04,
3,CNBC: Trump says talks between U.S. and Iran to resume in Pakistan on Monday,1612,Just in time before the futures open. -2% day cancelled for Monday?,2026-04-19 18:35:00,


## Step 8 - Inspect the latest top 10 WSB stocks

In [8]:
summary_view = summary_df[["ticker", "mention_count", "post_count", "avg_sentiment", "engagement", "latest_mention"]].copy()
summary_view["latest_mention"] = pd.to_datetime(summary_view["latest_mention"], utc=True)
summary_view

,ticker,mention_count,post_count,avg_sentiment,engagement,latest_mention
48,SPY,24,13,-0.155254,133.623249,2026-04-20 00:11:27+00:00
31,MSTR,7,4,0.347100,31.395614,2026-04-20 01:15:48+00:00
43,RKLB,10,2,0.066450,19.525002,2026-04-20 02:30:01+00:00
42,RACE,6,3,0.550033,33.992138,2026-04-18 03:58:35+00:00
2,AMD,6,3,0.147533,32.866942,2026-04-17 23:19:21+00:00
6,ASTS,6,3,-0.230967,26.778543,2026-04-20 02:30:01+00:00
50,TSLA,4,3,0.468600,26.855628,2026-04-18 10:58:22+00:00
41,QQQ,3,3,0.416500,31.854319,2026-04-19 14:21:18+00:00
37,NVDA,3,3,-0.361900,28.905978,2026-04-20 02:14:04+00:00
35,NFLX,5,2,0.129350,12.336443,2026-04-17 21:02:22+00:00


## Step 9 - Review sample mentions for the highest-ranked ticker

In [9]:
selected_ticker = str(summary_df.iloc[0]["ticker"])
selected_mentions_df = mentions_df.loc[mentions_df["ticker"].eq(selected_ticker)].copy()
selected_mentions_df["created_utc"] = pd.to_datetime(selected_mentions_df["created_utc"], utc=True)
print(f"Selected ticker: {selected_ticker}")
selected_mentions_df[["created_utc", "ticker", "mentions", "sentiment", "score", "num_comments", "title"]].head(20)

Selected ticker: SPY


,created_utc,ticker,mentions,sentiment,score,num_comments,title
6,2026-04-20 00:11:27+00:00,SPY,7,-0.9840,1011,462,Upcoming Stock Market Drop Will Be Epic Fury
45,2026-04-17 21:26:47+00:00,SPY,1,0.1327,91,74,How to martingale using stonks
52,2026-04-17 21:11:59+00:00,SPY,2,0.0000,89,45,Compounding ATM 0DTE SPY calls at open each day this week would have landed you a 62-bagger by end of week
55,2026-04-17 20:08:16+00:00,SPY,1,-0.7845,53,12,SPY Calls no Brain Activity
85,2026-04-17 15:14:12+00:00,SPY,1,0.5719,31,5,Sold early but cashed in on a nice SPY return
86,2026-04-17 14:48:47+00:00,SPY,1,0.5267,24,5,30k gain on SPY calls
112,2026-04-16 10:30:44+00:00,SPY,1,0.1531,2836,472,"Turned $700 into $70,000 in 3 days"
115,2026-04-16 08:10:02+00:00,SPY,1,-0.9876,1920,318,The last bear is me
166,2026-04-15 20:00:19+00:00,SPY,1,-0.6280,3981,331,SPY Officially Above 700 Dollars for the First Time
167,2026-04-15 19:29:57+00:00,SPY,1,0.0000,864,126,$700 SPY history in the making


## Step 10 - Final notebook summary

In [10]:
notebook_summary = {
    "data_source": source_name,
    "posts_fetched": int(len(posts_df)),
    "latest_top_ticker": selected_ticker,
    "top_10_tickers": summary_df["ticker"].tolist(),
    "output_dir": str(OUTPUT_DIR.resolve()),
    "legacy_csvs": [
        str((ROOT / "wsb_reddit_api_data.csv").resolve()),
        str((ROOT / "wsb_pushshift_data.csv").resolve()),
    ],
}
print(json.dumps(notebook_summary, indent=2))

{
  "data_source": "reddit_public_json",
  "posts_fetched": 166,
  "latest_top_ticker": "SPY",
  "top_10_tickers": [
    "SPY",
    "MSTR",
    "RKLB",
    "RACE",
    "AMD",
    "ASTS",
    "TSLA",
    "QQQ",
    "NVDA",
    "NFLX"
  ],
  "output_dir": "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\outputs\\latest_wsb_analysis",
  "legacy_csvs": [
    "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\wsb_reddit_api_data.csv",
    "C:\\Users\\big_j\\PycharmProjects\\Project2-Sentiment-analysis-WSbets\\wsb_pushshift_data.csv"
  ]
}
